# Task 1 — Build Your First RAG Application (End to End)

**Project:** Document Question-Answering RAG for *NovaCart*, a **fictional** e-commerce company
**Stack:** Python · Euron EURI (embeddings + LLM) · Qdrant (in-memory or Cloud) · SQLite or Supabase · FastAPI · LangSmith
**Reference:** Chapter 2 — *Building Your First RAG*

---

## What you will build

A system that reads a small company knowledge base (PDF, Word, Markdown, TXT), indexes it, and
answers questions with **citations** like `[S1]` that point back to the exact file and page.

```text
INGESTION  (runs once per document)
  file → load → extract text → normalize → chunk → embed (EURI) → store (Qdrant)
                                                                 ↘ register (SQLite/Supabase)

QUERY  (runs on every question)
  question → embed → similarity search → build context → grounded prompt → LLM (EURI)
           → answer + sources + timings
```

## How this notebook is organised

Every section is **one future module** from the chapter's *Section 17 — Suggested project structure*.
When we modularize, each section's code moves into the file shown in the right-hand column, with little or no change.

| # | Notebook section | Future module |
|---|---|---|
| 1 | Configuration, logging, tracing | `app/core/config.py`, `app/core/logging.py`, `app/core/tracing.py` |
| 2 | Data models | `app/models/document.py` |
| 3 | Sample dataset | `data/` (+ `make_dataset.py`) |
| 4 | Loaders and text extraction | `app/loaders/text_loader.py`, `pdf_loader.py`, `docx_loader.py` |
| 5 | Normalization and chunking | `app/rag/chunking.py` |
| 6 | Embeddings | `app/rag/embeddings.py` |
| 7 | Vector storage | `app/db/qdrant.py`, `app/rag/indexing.py` |
| 8 | Document registry | `app/db/registry.py` |
| 9 | Ingestion pipeline | `app/rag/indexing.py` |
| 10 | Similarity search | `app/rag/retrieval.py` |
| 11 | Context construction | `app/rag/context.py` |
| 12 | Prompt creation | `app/rag/prompting.py` |
| 13 | LLM generation | `app/rag/generation.py` |
| 14 | Answer + source attribution | `app/rag/pipeline.py` |
| 15 | FastAPI | `app/models/api.py`, `app/api/*.py`, `app/main.py` |
| 16 | Evaluation | `tests/`, `evaluation/` |
| 17 | Failure cases | *(teaching only)* |

> **Mentor principle:** understand the data flow by hand first. No LangChain, no LlamaIndex — just Python functions you can read.

## 0. Setup

### 0.1 Install dependencies

Run once in your virtual environment:

```bash
pip install -r requirements.txt
```

### 0.2 Create your `.env`

Copy `.env.example` to `.env` and fill in your values. Only `EURI_API_KEY` is required.
The **defaults need no accounts**: in-memory Qdrant + a local SQLite file. Two switches move you to the cloud:

| Switch | Default | Cloud option | Needs |
|---|---|---|---|
| `VECTOR_STORE` | `memory` | `cloud` → Qdrant Cloud | `QDRANT_URL`, `QDRANT_API_KEY` |
| `REGISTRY_BACKEND` | `sqlite` | `supabase` → Supabase table | `SUPABASE_URL`, `SUPABASE_KEY` + run `sql/supabase_registry.sql` once |
| `LANGSMITH_TRACING` | `false` | `true` → traces in LangSmith | `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT` |

Other settings:

| Variable | Default | Meaning |
|---|---|---|
| `EURI_API_KEY` | — | **Required.** Your Euron key |
| `EURI_BASE_URL` | `https://api.euron.one/api/v1` | Any OpenAI-compatible endpoint works |
| `EURI_CHAT_MODEL` | `gpt-4.1-mini` | Model that writes the answer |
| `EURI_EMBEDDING_MODEL` | `text-embedding-3-small` | Model that creates vectors |

> Why switches instead of "use the cloud if a URL is set"? Because you can keep all credentials in `.env`
> and still flip back to the offline defaults with one word — handy in class and in tests.

> Never commit `.env` to GitHub. It is already listed in `.gitignore`.

In [ ]:
# Uncomment to install inside the notebook instead of the terminal
# %pip install -r requirements.txt

---
# 1. Configuration, logging and tracing
**→ future modules:** `app/core/config.py`, `app/core/logging.py`, `app/core/tracing.py`

Every setting lives in **one** object. Why? The embedding model, the vector size and the
collection name must agree everywhere. If they are spread across files they drift apart, and
the error (`expected dim 1536, got 768`) shows up far from its cause.

In [41]:
import os
import re
import time
import uuid
import json
import hashlib
import logging
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Optional, Iterable, Tuple

import pandas as pd
from dotenv import load_dotenv

load_dotenv()   # reads key=value lines from .env into environment variables

pd.set_option("display.max_colwidth", 120)

In [42]:
@dataclass
class Settings:
    # --- EURI (OpenAI-compatible gateway) ---
    euri_api_key: Optional[str] = os.getenv("EURI_API_KEY")
    euri_base_url: str = os.getenv("EURI_BASE_URL", "https://api.euron.one/api/v1/euri")
    chat_model: str = os.getenv("EURI_CHAT_MODEL", "gpt-4.1-mini")
    embedding_model: str = os.getenv("EURI_EMBEDDING_MODEL", "text-embedding-3-small")

    # --- Vector store: "memory" (default) or "cloud" (Qdrant Cloud) ---
    vector_store: str = os.getenv("VECTOR_STORE", "memory").strip().lower()
    qdrant_url: Optional[str] = os.getenv("QDRANT_URL") or None
    qdrant_api_key: Optional[str] = os.getenv("QDRANT_API_KEY") or None
    collection_name: str = os.getenv("QDRANT_COLLECTION", "novacart_first_rag")

    # --- Relational registry: "sqlite" (default) or "supabase" ---
    registry_backend: str = os.getenv("REGISTRY_BACKEND", "sqlite").strip().lower()
    sqlite_path: str = os.getenv("SQLITE_PATH", "rag_registry.db")
    # DATABASE_URL / DATABASE_KEY are accepted as older names for the same two values
    supabase_url: Optional[str] = os.getenv("SUPABASE_URL") or os.getenv("DATABASE_URL") or None
    supabase_key: Optional[str] = os.getenv("SUPABASE_KEY") or os.getenv("DATABASE_KEY") or None

    # --- LangSmith tracing (the langsmith SDK reads these env vars itself) ---
    tracing_enabled: bool = os.getenv("LANGSMITH_TRACING", "false").strip().lower() == "true"
    langsmith_project: str = os.getenv("LANGSMITH_PROJECT", "default")

    # --- RAG knobs ---
    chunk_size: int = int(os.getenv("CHUNK_SIZE", 120))          # words per chunk
    chunk_overlap: int = int(os.getenv("CHUNK_OVERLAP", 20))     # words repeated between chunks
    embed_batch_size: int = int(os.getenv("EMBED_BATCH_SIZE", 64))
    top_k: int = int(os.getenv("TOP_K", 5))
    max_context_chars: int = int(os.getenv("MAX_CONTEXT_CHARS", 12000))
    temperature: float = float(os.getenv("LLM_TEMPERATURE", 0.1))

    # --- Files ---
    data_dir: Path = Path(os.getenv("DATA_DIR", "data"))
    upload_dir: Path = Path(os.getenv("UPLOAD_DIR", "data/uploads"))
    allowed_extensions: Tuple[str, ...] = (".txt", ".md", ".pdf", ".docx")


settings = Settings()

# Fail early with a clear message instead of a confusing error 50 cells later
assert settings.vector_store in ("memory", "cloud"), "VECTOR_STORE must be 'memory' or 'cloud'"
assert settings.registry_backend in ("sqlite", "supabase"), "REGISTRY_BACKEND must be 'sqlite' or 'supabase'"
if settings.vector_store == "cloud" and not (settings.qdrant_url and settings.qdrant_api_key):
    raise ValueError("VECTOR_STORE=cloud needs QDRANT_URL and QDRANT_API_KEY in .env")
if settings.registry_backend == "supabase" and not (settings.supabase_url and settings.supabase_key):
    raise ValueError("REGISTRY_BACKEND=supabase needs SUPABASE_URL and SUPABASE_KEY in .env")

# Print the config WITHOUT leaking the secret
print("EURI base URL      :", settings.euri_base_url)
print("EURI key configured:", bool(settings.euri_api_key))
print("Chat model         :", settings.chat_model)
print("Embedding model    :", settings.embedding_model)
print("Vector store       :", "Qdrant Cloud" if settings.vector_store == "cloud" else "in-memory Qdrant")
print("Registry           :", "Supabase" if settings.registry_backend == "supabase" else f"SQLite ({settings.sqlite_path})")
print("LangSmith tracing  :", f"ON (project '{settings.langsmith_project}')" if settings.tracing_enabled else "off")
print("Chunking           :", settings.chunk_size, "words, overlap", settings.chunk_overlap)

EURI base URL      : https://api.euron.one/api/v1/euri
EURI key configured: True
Chat model         : gpt-4.1-mini
Embedding model    : text-embedding-3-small
Vector store       : Qdrant Cloud
Registry           : Supabase
LangSmith tracing  : ON (project 'novacart_first_rag_project')
Chunking           : 120 words, overlap 20


In [43]:
# app/core/logging.py — one consistent log format for the whole app
def get_logger(name: str = "first_rag") -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:                      # avoid duplicate handlers on re-run
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%H:%M:%S"))
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
    return logger

log = get_logger()
log.info("Logger ready")

09:26:03 | INFO | Logger ready


### 1.1 Tracing with LangSmith
**→ future module:** `app/core/tracing.py`

Logs tell you *that* something happened. A **trace** shows the whole request as a tree — which chunks
were retrieved, the exact prompt sent, the answer, tokens and latency of every step:

```text
rag_pipeline                     (chain)      1.9 s
├── similarity_search            (retriever)  0.3 s
│   └── embed_texts              (embedding)
├── build_context                (chain)
└── generate_answer              (chain)
    └── ChatOpenAI               (llm)        1.5 s   prompt + completion tokens
```

We use two tools from the `langsmith` SDK:

- `@traceable` — put it on any function and it becomes a step in the trace.
- `wrap_openai` — wraps the OpenAI client so every LLM call is logged with its messages and token usage.

When `LANGSMITH_TRACING` is not `true`, both are **no-ops** — the code runs exactly the same, just untraced.

In [44]:
from langsmith import traceable
from langsmith.wrappers import wrap_openai


def summarize_vectors(output: Any) -> Dict[str, Any]:
    # Don't upload thousands of floats to LangSmith; log the shape instead
    vectors = output.get("output", output) if isinstance(output, dict) else output
    return {"num_vectors": len(vectors), "dim": len(vectors[0]) if vectors else 0}


def summarize_hits(output: Any) -> Dict[str, Any]:
    # Keep retriever traces readable: source, page, score and a text preview per hit
    hits = output.get("output", output) if isinstance(output, dict) else output
    return {"hits": [{"rank": h["rank"], "score": round(h["score"], 4), "source": h["payload"]["source"],
                      "page": h["payload"].get("page"), "text": h["payload"]["text"][:300]} for h in hits]}


print("Tracing:", f"ON -> https://smith.langchain.com (project '{settings.langsmith_project}')"
      if settings.tracing_enabled else "off (set LANGSMITH_TRACING=true to enable)")

Tracing: ON -> https://smith.langchain.com (project 'novacart_first_rag_project')


---
# 2. Data models
**→ future module:** `app/models/document.py`

A RAG pipeline passes the same few objects around. Making them explicit (instead of loose
dicts) makes the pipeline much easier to debug.

| Object | What it is |
|---|---|
| `Document` | normalized text of one file (or one PDF **page**) + metadata |
| `Chunk` | the smallest unit the retriever can return |

**About IDs.** IDs are **deterministic** (hashes), not random:

- `document_id` = hash of the **file name** → re-uploading `refund_policy.md` updates the same document instead of creating a duplicate.
- `chunk_id` = hash of `document_id + page + chunk_index` → re-ingesting overwrites the same Qdrant points.

> The chapter's reference code used `id=1, 2, 3…` for Qdrant points. That silently **overwrites** the
> previous file's points when you ingest a second file. Deterministic IDs fix it.

In [45]:
def stable_id(value: str, prefix: str = "doc") -> str:
    # Same input -> same ID, every time, on every machine
    digest = hashlib.sha256(value.encode("utf-8")).hexdigest()[:16]
    return f"{prefix}-{digest}"


def content_hash(text: str) -> str:
    # Fingerprint of the content: tells us whether a file actually changed
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


@dataclass
class Document:
    document_id: str
    text: str
    source: str                      # file path or URL
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class Chunk:
    chunk_id: str
    document_id: str
    text: str
    source: str
    metadata: Dict[str, Any] = field(default_factory=dict)


print(asdict(Document(document_id=stable_id("demo.txt"), text="RAG connects LLMs to knowledge.", source="demo.txt")))

{'document_id': 'doc-90ac49d3367b3295', 'text': 'RAG connects LLMs to knowledge.', 'source': 'demo.txt', 'metadata': {}}


---
# 3. Sample dataset
**→ folder:** `data/` (generated by `make_dataset.py`)

We use a small, fully **fictional** knowledge base for *NovaCart*, an imaginary Indian e-commerce
company. One file per supported format, so every loader is exercised:

| File | Format | Topics |
|---|---|---|
| `novacart_employee_handbook.pdf` | PDF, 4 pages | working hours, leave, hybrid work, expenses |
| `refund_policy.md` | Markdown | return windows, refund timelines, pickup fee |
| `novacart_plus_terms.docx` | Word | membership pricing, benefits, cancellation |
| `it_security_faq.txt` | Text | passwords, MFA, phishing, lost laptop |
| `archive/refund_policy_2025.md` | Markdown | **old** version — only used in the failure demos |

In [46]:
# Build the dataset if it is not there yet
if not (settings.data_dir / "refund_policy.md").exists():
    "Building the dataset as it does not exist yet..."
    import make_dataset
    make_dataset.main()

for f in sorted(settings.data_dir.rglob("*")):
    "Listing all files in the data directory"
    if f.is_file() and "uploads" not in f.parts:
        print(f"{str(f.relative_to(settings.data_dir)):40s} {f.stat().st_size:>8,} bytes")

archive\refund_policy_2025.md                 336 bytes
it_security_faq.txt                         1,266 bytes
novacart_employee_handbook.pdf              4,991 bytes
novacart_plus_terms.docx                   37,264 bytes
refund_policy.md                            1,238 bytes


---
# 4. Document loading and text extraction
**→ future modules:** `app/loaders/text_loader.py`, `app/loaders/pdf_loader.py`, `app/loaders/docx_loader.py`

Two different jobs that are easy to confuse:

- **Loading** — get the raw file and its metadata (name, size, type).
- **Extraction** — turn the format into plain text. A PDF is *not* text: it is pages of drawing
  commands, fonts and positions. Extraction reconstructs the words.

For a first RAG we keep extraction simple. Things we are *ignoring for now* (Advanced RAG topics):
scanned PDFs (need OCR), tables, images, multi-column reading order.

### 4.1 Text and Markdown loader
Markdown is loaded as plain text. Headings like `## Refund timelines` stay in the text, which
actually helps retrieval because the heading words get embedded with the paragraph.

In [47]:
def load_text_file(path: str | Path) -> List[Document]:
    p = Path(path)
    text = p.read_text(encoding="utf-8")
    return [
        Document(
            document_id=stable_id(p.name),
            text=text,
            source=p.name,
            metadata={
                "filename": p.name,
                "source_type": p.suffix.lower().lstrip("."),   # "txt" or "md"
                "page": None,
                "size_bytes": p.stat().st_size,
            },
        )
    ]


docs = load_text_file(settings.data_dir / "it_security_faq.txt")
print(docs[0].metadata)
print(docs[0].text[:300])

{'filename': 'it_security_faq.txt', 'source_type': 'txt', 'page': None, 'size_bytes': 1266}
NovaCart IT Security FAQ (internal, fictional)

Q: How often must I change my password?
A: Passwords must be changed every 90 days. They need at least 14 characters and cannot reuse
any of your last 5 passwords.

Q: Is multi-factor authentication mandatory?
A: Yes. Multi-factor authentication (MFA) 


In [11]:
print(docs[0].text)

NovaCart IT Security FAQ (internal, fictional)

Q: How often must I change my password?
A: Passwords must be changed every 90 days. They need at least 14 characters and cannot reuse
any of your last 5 passwords.

Q: Is multi-factor authentication mandatory?
A: Yes. Multi-factor authentication (MFA) is mandatory for email, the VPN, GitHub and every
production system. Use the NovaCart Authenticator app; SMS codes are not allowed.

Q: What should I do with a suspicious email?
A: Do not click links or open attachments. Use the "Report phishing" button in your mail client
or forward the email to security@novacart.example.

Q: My laptop was lost or stolen. What now?
A: Report it to the IT service desk within 24 hours by calling extension 4357 or raising a
ticket in the IT portal. IT will remotely lock and wipe the device.

Q: Can I install software on my company laptop?
A: Only from the NovaCart Software Center. Anything else needs an approved IT ticket.

Q: Can I use personal USB drives?
A:

In [48]:
docs

[Document(document_id='doc-d12df8b3b237c572', text='NovaCart IT Security FAQ (internal, fictional)\n\nQ: How often must I change my password?\nA: Passwords must be changed every 90 days. They need at least 14 characters and cannot reuse\nany of your last 5 passwords.\n\nQ: Is multi-factor authentication mandatory?\nA: Yes. Multi-factor authentication (MFA) is mandatory for email, the VPN, GitHub and every\nproduction system. Use the NovaCart Authenticator app; SMS codes are not allowed.\n\nQ: What should I do with a suspicious email?\nA: Do not click links or open attachments. Use the "Report phishing" button in your mail client\nor forward the email to security@novacart.example.\n\nQ: My laptop was lost or stolen. What now?\nA: Report it to the IT service desk within 24 hours by calling extension 4357 or raising a\nticket in the IT portal. IT will remotely lock and wipe the device.\n\nQ: Can I install software on my company laptop?\nA: Only from the NovaCart Software Center. Anything 

### 4.2 PDF loader — one `Document` per page
We keep **one Document per page** so every chunk remembers its page number. That is what lets us
cite *"handbook, page 2"* later. All pages share the file's `document_id`.

In [49]:
import pymupdf   # PyMuPDF (older code imports it as "fitz")

def load_pdf(path: str | Path) -> List[Document]:
    p = Path(path)
    document_id = stable_id(p.name)
    pages = []

    with pymupdf.open(p) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = page.get_text("text")
            if not text.strip():
                # Empty page -> probably a scanned image. OCR is out of scope for now.
                log.warning("No text on %s page %s (scanned?)", p.name, page_number)
                continue
            pages.append(
                Document(
                    document_id=document_id,
                    text=text,
                    source=p.name,
                    metadata={
                        "filename": p.name,
                        "source_type": "pdf",
                        "page": page_number,
                        "total_pages": len(pdf),
                    },
                )
            )
    return pages


pdf_pages = load_pdf(settings.data_dir / "novacart_employee_handbook.pdf")
print(f"{len(pdf_pages)} pages loaded")
for d in pdf_pages:
    print(f"page {d.metadata['page']}: {d.text[:80]!r}")

4 pages loaded
page 1: 'NovaCart Employee Handbook 2026 - Section 1: Working at NovaCart\nNovaCart is a f'
page 2: 'Section 2: Leave Policy\nEvery confirmed employee receives 24 days of paid annual'
page 3: 'Section 3: Hybrid and Remote Work\nNovaCart follows a hybrid model: employees wor'
page 4: 'Section 4: Expenses and Code of Conduct\nBusiness expenses must be submitted with'


In [11]:
pdf_pages

[Document(document_id='doc-9e69091949d57ccb', text='NovaCart Employee Handbook 2026 - Section 1: Working at NovaCart\nNovaCart is a fictional e-commerce company headquartered in Bengaluru with offices in Pune and\nHyderabad.\nThis handbook applies to all full-time employees from 1 January 2026.\nStandard working hours are 9:30 AM to 6:30 PM, Monday to Friday, including a one-hour lunch break.\nCore collaboration hours, when every employee must be reachable, are 11:00 AM to 4:00 PM IST.\nNew employees complete a 6-month probation period. During probation the notice period is 15 days;\nafter confirmation the notice period is 60 days.\nEvery new joiner is assigned an onboarding buddy for the first 30 days. The buddy helps with tools,\nteam introductions and the first production deployment.\n', source='novacart_employee_handbook.pdf', metadata={'filename': 'novacart_employee_handbook.pdf', 'source_type': 'pdf', 'page': 1, 'total_pages': 4}),
 Document(document_id='doc-9e69091949d57ccb', te

In [51]:
type(pdf_pages)

list

In [15]:
type(pdf_pages[0])

__main__.Document

### 4.3 DOCX loader
`python-docx` gives us paragraphs. Headings are paragraphs too, so they stay in reading order.

In [16]:
from docx import Document as DocxDocument

def load_docx(path: str | Path) -> List[Document]:
    p = Path(path)
    doc = DocxDocument(p)
    paragraphs = [para.text for para in doc.paragraphs if para.text.strip()]
    return [
        Document(
            document_id=stable_id(p.name),
            text="\n".join(paragraphs),
            source=p.name,
            metadata={"filename": p.name, "source_type": "docx", "page": None},
        )
    ]


docx_docs = load_docx(settings.data_dir / "novacart_plus_terms.docx")
print(docx_docs[0].text[:400])

NovaCart Plus Membership Terms
Effective date: 1 February 2026. NovaCart Plus is a paid membership programme.
1. Plans and pricing
NovaCart Plus costs Rs 999 per year or Rs 129 per month. The yearly plan includes a 30-day free trial for first-time members. Prices include GST.
2. Member benefits
Plus members get free delivery on every order with no minimum order value, early access to sale events 2


In [13]:
docx_docs

[Document(document_id='doc-7642d59ce7a9ffbe', text='NovaCart Plus Membership Terms\nEffective date: 1 February 2026. NovaCart Plus is a paid membership programme.\n1. Plans and pricing\nNovaCart Plus costs Rs 999 per year or Rs 129 per month. The yearly plan includes a 30-day free trial for first-time members. Prices include GST.\n2. Member benefits\nPlus members get free delivery on every order with no minimum order value, early access to sale events 24 hours before other customers, 5% extra cashback on NovaCart Pay, and free return pickups. Non-members get free delivery only on orders above Rs 499.\n3. Cancellation and refunds\nMembers can cancel the yearly plan within 14 days of purchase for a full refund, but only if no Plus benefit has been used, such as a free delivery or early sale access. If a benefit has been used, the membership is cancelled at the end of the current billing period and no refund is issued. Monthly plans can be cancelled at any time and stop at the end of the 

In [17]:
len(docx_docs)

1

In [14]:
docx_docs[0].metadata

{'filename': 'novacart_plus_terms.docx', 'source_type': 'docx', 'page': None}

### 4.4 Web page loader (bonus)
Same idea for HTML: download, strip `<script>`/`<style>`, keep visible text. We will not index
web pages in this project, but the loader is here so you can try it.

In [52]:
import requests
from bs4 import BeautifulSoup

# Many sites (Wikipedia included) return 403 Forbidden to the default "python-requests/x.y"
# User-Agent. Identify your client honestly: app name/version + a way to contact you.
HTTP_HEADERS = {
    "User-Agent": "NovaCartFirstRAG/1.0 (educational RAG project; contact: your-email@example.com)",
    "Accept-Language": "en",
}

def load_webpage(url: str) -> List[Document]:
    response = requests.get(url, headers=HTTP_HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = "\n".join(line.strip() for line in soup.get_text("\n").splitlines() if line.strip())
    title = soup.title.string.strip() if soup.title and soup.title.string else None

    return [Document(document_id=stable_id(url), text=text, source=url,
                     metadata={"source_type": "web", "url": url, "title": title, "page": None})]

# Example (needs internet):
web_docs = load_webpage("https://en.wikipedia.org/wiki/Retrieval-augmented_generation")
print(web_docs[0].metadata["title"], "|", len(web_docs[0].text), "chars")
print(web_docs[0].text[:300])

Retrieval-augmented generation - Wikipedia | 24756 chars
Retrieval-augmented generation - Wikipedia
Jump to content
Main menu
Main menu
move to sidebar
hide
Navigation
Main page
Contents
Current events
Random article
About Wikipedia
Contact us
Contribute
Help
Learn to edit
Community portal
Recent changes
Upload file
Special pages
Search
Search
Appearance



### 4.5 One entry point for every file type
The rest of the pipeline should not care about formats. `load_document()` picks the right loader
from the file extension; `load_directory()` loads a whole folder.

In [19]:
LOADERS = {
    ".txt": load_text_file,
    ".md": load_text_file,
    ".pdf": load_pdf,
    ".docx": load_docx,
}

def validate_document_filename(filename: str) -> str:
    suffix = Path(filename).suffix.lower()
    if suffix not in settings.allowed_extensions:
        raise ValueError(f"Unsupported file type '{suffix}'. Allowed: {list(settings.allowed_extensions)}")
    return suffix


def load_document(path: str | Path) -> List[Document]:
    suffix = validate_document_filename(str(path))
    return LOADERS[suffix](path)


def load_directory(folder: str | Path) -> List[Document]:
    # Top-level files only: data/archive/ and data/uploads/ are NOT picked up
    documents = []
    for f in sorted(Path(folder).iterdir()):
        if f.is_file() and f.suffix.lower() in settings.allowed_extensions:
            documents.extend(load_document(f))
    return documents


all_docs = load_directory(settings.data_dir)
pd.DataFrame([
    {"source": d.source, "type": d.metadata["source_type"], "page": d.metadata.get("page"),
     "chars": len(d.text), "document_id": d.document_id}
    for d in all_docs
])

,source,type,page,chars,document_id
0,it_security_faq.txt,txt,NaN,1266,doc-d12df8b3b237c572
1,novacart_employee_handbook.pdf,pdf,1.0,737,doc-9e69091949d57ccb
2,novacart_employee_handbook.pdf,pdf,2.0,758,doc-9e69091949d57ccb
3,novacart_employee_handbook.pdf,pdf,3.0,659,doc-9e69091949d57ccb
4,novacart_employee_handbook.pdf,pdf,4.0,658,doc-9e69091949d57ccb
5,novacart_plus_terms.docx,docx,NaN,1315,doc-7642d59ce7a9ffbe
6,refund_policy.md,md,NaN,1238,doc-a832f9e34e43beb7


---
# 5. Normalization and chunking
**→ future module:** `app/rag/chunking.py`

### 5.1 Normalization
Light, **non-destructive** cleanup before chunking: unify line endings, collapse repeated
spaces and blank lines. We do **not** strip punctuation or headings — they carry meaning.

In [53]:
def normalize_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")   # Windows/Mac line endings -> \n
    text = re.sub(r"[ \t]+", " ", text)                       # many spaces/tabs -> one space
    text = re.sub(r"\n{3,}", "\n\n", text)                    # 3+ blank lines -> 1 blank line
    return text.strip()


print(normalize_text("RAG    retrieves   documents.\r\n\r\n\r\n\r\nIt grounds answers."))

RAG retrieves documents.

It grounds answers.


In [54]:
print("RAG    retrieves   documents.\r\n\r\n\r\n\r\nIt grounds answers.")

RAG    retrieves   documents.



It grounds answers.


### 5.2 Chunking — why?
- **Whole documents are too big.** A 4-page handbook covers leave, remote work *and* expenses.
  One vector for all of it matches no question well.
- **Tiny chunks lose context.** *"Cancel within 14 days"* and *"only if no benefit was used"* may land in
  different chunks, and the retriever returns half the rule.

Chunking is a trade-off between **precision** (small) and **context** (large).
We use a simple **word window with overlap**:

```text
words:   w1 ........ w120
                  w101 ........ w220        <- 20 words overlap
                               w201 ........ w320
```

In [55]:
def chunk_document(document: Document, chunk_size: int = 120, overlap: int = 20) -> List[Chunk]:
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    words = normalize_text(document.text).split()
    page = document.metadata.get("page")
    chunks: List[Chunk] = []
    start, chunk_index = 0, 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        text = " ".join(words[start:end])

        chunks.append(
            Chunk(
                # document + page + position -> stable and unique
                chunk_id=stable_id(f"{document.document_id}::p{page}::c{chunk_index}", prefix="chunk"),
                document_id=document.document_id,
                text=text,
                source=document.source,
                metadata={**document.metadata, "chunk_index": chunk_index,
                          "word_start": start, "word_end": end},
            )
        )

        if end == len(words):
            break
        start = end - overlap          # step back by `overlap` words
        chunk_index += 1

    return chunks


def chunk_documents(documents: List[Document], chunk_size: int, overlap: int) -> List[Chunk]:
    all_chunks = []
    for d in documents:
        all_chunks.extend(chunk_document(d, chunk_size, overlap))
    return all_chunks

In [56]:
# Inspect the chunks for the Plus membership terms
plus_chunks = chunk_document(docx_docs[0], chunk_size=settings.chunk_size, overlap=settings.chunk_overlap)
pd.DataFrame([{"chunk_index": c.metadata["chunk_index"], "words": len(c.text.split()),
               "start": c.metadata["word_start"], "end": c.metadata["word_end"], "text": c.text}
              for c in plus_chunks])

,chunk_index,words,start,end,text
0,0,120,0,120,NovaCart Plus Membership Terms Effective date: 1 February 2026. NovaCart Plus is a paid membership programme. 1. Pla...
1,1,120,100,220,"yearly plan within 14 days of purchase for a full refund, but only if no Plus benefit has been used, such as a free ..."
2,2,26,200,226,A yearly Plus member can share benefits with up to 3 family members living at the same delivery address. Monthly pla...


In [64]:
print(docx_docs[0].text)

NovaCart Plus Membership Terms
Effective date: 1 February 2026. NovaCart Plus is a paid membership programme.
1. Plans and pricing
NovaCart Plus costs Rs 999 per year or Rs 129 per month. The yearly plan includes a 30-day free trial for first-time members. Prices include GST.
2. Member benefits
Plus members get free delivery on every order with no minimum order value, early access to sale events 24 hours before other customers, 5% extra cashback on NovaCart Pay, and free return pickups. Non-members get free delivery only on orders above Rs 499.
3. Cancellation and refunds
Members can cancel the yearly plan within 14 days of purchase for a full refund, but only if no Plus benefit has been used, such as a free delivery or early sale access. If a benefit has been used, the membership is cancelled at the end of the current billing period and no refund is issued. Monthly plans can be cancelled at any time and stop at the end of the month.
4. Auto-renewal
Memberships renew automatically. Nov

In [65]:
# Experiment: how does chunk size change the number of chunks across the whole corpus?
rows = []
for size, ov in [(40, 5), (80, 15), (120, 20), (250, 40), (1000, 100)]:
    cs = chunk_documents(all_docs, size, ov)
    rows.append({"chunk_size": size, "overlap": ov, "num_chunks": len(cs),
                 "avg_words": round(sum(len(c.text.split()) for c in cs) / len(cs), 1)})
pd.DataFrame(rows)

,chunk_size,overlap,num_chunks,avg_words
0,40,5,35,36.0
1,80,15,19,68.4
2,120,20,12,101.7
3,250,40,7,160.0
4,1000,100,7,160.0


**Reading the table:** with 1000-word chunks each file (or PDF page) becomes a single chunk — no
precision at all. There is no universal best size; it depends on your documents and questions.
We start at **120 words / 20 overlap** and test other values in Section 17.

---
# 6. Embeddings
**→ future module:** `app/rag/embeddings.py`

An embedding model turns text into a vector of numbers: $f(\text{text}) \rightarrow \mathbf{v} \in \mathbb{R}^d$.
Texts with similar **meaning** land close together, even with different words:

```text
"How can I cancel my plan?"          ─┐  close
"How do I terminate my subscription?" ─┘
"What is the office Wi-Fi password?"      far away
```

EURI is **OpenAI-compatible**, so we use the official `openai` Python client and just change `base_url`.

In [66]:
from openai import OpenAI

_client: Optional[OpenAI] = None

def get_llm_client() -> OpenAI:
    # One shared client for embeddings and chat (created lazily on first use)
    global _client
    if _client is None:
        if not settings.euri_api_key:
            raise RuntimeError("EURI_API_KEY is missing. Add it to your .env file and restart the kernel.")
        _client = OpenAI(api_key=settings.euri_api_key, base_url=settings.euri_base_url)
        if settings.tracing_enabled:
            _client = wrap_openai(_client)       # every chat call now shows up in LangSmith
    return _client

### 6.1 Batching, retries and caching
Three habits every ingestion pipeline needs from day one:

1. **Batch** — send 64 texts per request, not 1. Fewer calls, faster, cheaper.
2. **Retry** — networks fail and APIs rate-limit. Wait and try again (exponential backoff: 1s, 2s, 4s).
3. **Cache** — the same text always gives the same vector. Don't pay twice for it.

In [67]:
def batched(items: List[Any], batch_size: int) -> Iterable[List[Any]]:
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


_embedding_cache: Dict[str, List[float]] = {}   # content hash -> vector (in memory only)

def _embed_batch_with_retry(texts: List[str], max_retries: int = 3) -> List[List[float]]:
    for attempt in range(max_retries):
        try:
            response = get_llm_client().embeddings.create(model=settings.embedding_model, input=texts)
            return [item.embedding for item in response.data]
        except Exception as exc:
            if attempt == max_retries - 1:
                raise
            wait = 2 ** attempt
            log.warning("Embedding call failed (%s). Retrying in %ss...", exc, wait)
            time.sleep(wait)


@traceable(run_type="embedding", name="embed_texts", process_outputs=summarize_vectors)
def embed_texts(texts: List[str], batch_size: Optional[int] = None) -> List[List[float]]:
    batch_size = batch_size or settings.embed_batch_size
    keys = [content_hash(f"{settings.embedding_model}::{t}") for t in texts]

    # Only send texts we have never embedded before
    missing = [(k, t) for k, t in zip(keys, texts) if k not in _embedding_cache]
    unique_missing = list(dict(missing).items())            # de-duplicate identical texts

    for batch in batched(unique_missing, batch_size):
        vectors = _embed_batch_with_retry([t for _, t in batch])
        for (k, _), v in zip(batch, vectors):
            _embedding_cache[k] = v

    return [_embedding_cache[k] for k in keys]

In [26]:
import numpy as np

def cosine(a: List[float], b: List[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


probe = [
    "How can I cancel my plan?",
    "How do I terminate my subscription?",
    "What is the office Wi-Fi password?",
]
vectors = embed_texts(probe)
print("Embedding dimension:", len(vectors[0]))
print(f"cancel vs terminate : {cosine(vectors[0], vectors[1]):.3f}   <- same meaning")
print(f"cancel vs wifi      : {cosine(vectors[0], vectors[2]):.3f}   <- unrelated")

Embedding dimension: 1536
cancel vs terminate : 0.706   <- same meaning
cancel vs wifi      : 0.135   <- unrelated


> **Rule:** questions and documents **must** use the same embedding model. Vectors from two different
> models live in two different spaces — comparing them is meaningless.

---
# 7. Vector storage with Qdrant
**→ future modules:** `app/db/qdrant.py` (connection) and `app/rag/indexing.py` (writing points)

Qdrant stores **points**. A point = `id` + `vector` + `payload` (any JSON metadata).
The payload is what makes RAG trustworthy — it carries the text, the file name and the page used for citations and filtering.

```json
{"chunk_id": "chunk-3f2a…", "document_id": "doc-91c0…", "source": "novacart_employee_handbook.pdf",
 "page": 2, "chunk_index": 0, "source_type": "pdf", "text": "Every confirmed employee receives 24 days…"}
```

**Where does Qdrant run?** Controlled by `VECTOR_STORE`:

| `VECTOR_STORE` | Where | Survives a restart? | Use it for |
|---|---|---|---|
| `memory` *(default)* | inside this Python process | no | learning, quick experiments, tests |
| `cloud` | Qdrant Cloud (`QDRANT_URL` + `QDRANT_API_KEY`) | yes | the API, sharing, production |

The rest of the code never checks which one it is — `QdrantClient` has the same methods either way.
That is the payoff of hiding the choice behind **one** function.

> Local Docker works the same way: `docker run -p 6333:6333 qdrant/qdrant`, then
> `VECTOR_STORE=cloud` and `QDRANT_URL=http://localhost:6333` (any non-empty API key).

In [27]:
from qdrant_client import QdrantClient
from qdrant_client.models import (Distance, VectorParams, PointStruct,
                                  Filter, FieldCondition, MatchValue, FilterSelector)

def get_qdrant_client() -> QdrantClient:
    if settings.vector_store == "cloud":
        # timeout=60: the first calls to a cloud cluster can be slow while it wakes up
        return QdrantClient(url=settings.qdrant_url, api_key=settings.qdrant_api_key, timeout=60)
    return QdrantClient(":memory:")      # lives only as long as this kernel


qdrant = get_qdrant_client()
print("Vector store:", settings.vector_store)
print("Collections :", [c.name for c in qdrant.get_collections().collections])

Vector store: cloud
Collections : ['agentic_rag_task1_text_rag', 'agenticragmultimodel', 'agenticragmultimodel_v1', 'euron_agentic_rag_class', 'euron_agentic_rag_class_new', 'euron_agentic_rag_demo', 'euron_crag_ingestion_v2', 'euron_crag_ingestion_v3', 'multi_format_knowledge', 'novacart_first_rag', 'visual_data_vec']


In [29]:
def ensure_collection(client: QdrantClient, collection_name: str, vector_size: int) -> None:
    if client.collection_exists(collection_name):
        existing_size = client.get_collection(collection_name).config.params.vectors.size
        if existing_size != vector_size:
            raise ValueError(
                f"Collection '{collection_name}' has vectors of size {existing_size}, but the embedding "
                f"model returns {vector_size}. Use a new collection name or delete the old one."
            )
    else:
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
        )
        log.info("Created collection '%s' (dim=%s, cosine)", collection_name, vector_size)

    # Payload indexes make metadata filters fast. Qdrant Cloud can REFUSE to filter on a field
    # without one ("Index required but not found"). Re-creating an existing index is harmless.
    if settings.vector_store == "cloud":
        for key in ("document_id", "source"):
            client.create_payload_index(collection_name, field_name=key, field_schema="keyword")


def point_id_for(chunk_id: str) -> str:
    # Qdrant IDs must be an integer or a UUID -> turn our chunk_id into a stable UUID
    return str(uuid.uuid5(uuid.NAMESPACE_URL, chunk_id))


def delete_document_points(client: QdrantClient, collection_name: str, document_id: str) -> None:
    # Remove old chunks before re-indexing a changed document (otherwise stale chunks linger)
    if client.collection_exists(collection_name):
        client.delete(
            collection_name=collection_name,
            points_selector=FilterSelector(filter=Filter(
                must=[FieldCondition(key="document_id", match=MatchValue(value=document_id))]
            )),
        )


def index_chunks(client: QdrantClient, collection_name: str,
                 chunks: List[Chunk], embeddings: List[List[float]]) -> int:
    if len(chunks) != len(embeddings):
        raise ValueError("Chunks and embeddings must have equal length.")
    if not chunks:
        return 0

    ensure_collection(client, collection_name, vector_size=len(embeddings[0]))

    points = [
        PointStruct(
            id=point_id_for(chunk.chunk_id),
            vector=vector,
            payload={"chunk_id": chunk.chunk_id, "document_id": chunk.document_id,
                     "source": chunk.source, "text": chunk.text, **chunk.metadata},
        )
        for chunk, vector in zip(chunks, embeddings)
    ]

    for batch in batched(points, 256):             # upsert in batches too
        client.upsert(collection_name=collection_name, points=batch)
    return len(points)


def count_document_points(client: QdrantClient, collection_name: str, document_id: str) -> int:
    if not client.collection_exists(collection_name):
        return 0
    return client.count(collection_name, exact=True, count_filter=Filter(
        must=[FieldCondition(key="document_id", match=MatchValue(value=document_id))])).count